# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ishigupgta1234-ux/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
%pip install -q duckdb

import duckdb
from google.colab import userdata

token = userdata.get('HF_TOKEN')
con = duckdb.connect()
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{token}')")

rel = "hf://datasets/FlyRank/internship-warehouse"

print(con.sql(f"SELECT COUNT(*) FROM read_parquet('{rel}/fact_content_daily_performance/**/*.parquet')"))

grain_check = con.sql(f"""
    SELECT MAX(n) AS max_rows_per_key
    FROM (
        SELECT report_date, client_hash_id, content_hash_id, COUNT(*) AS n
        FROM read_parquet('{rel}/fact_content_daily_performance/**/*.parquet')
        WHERE month = '2026-03'
        GROUP BY report_date, client_hash_id, content_hash_id
    )
""")
print(grain_check)  # should print 1 — proves the grain claim above

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌──────────────┐
│ count_star() │
│    int64     │
├──────────────┤
│     78835655 │
└──────────────┘



FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌──────────────────┐
│ max_rows_per_key │
│      int64       │
├──────────────────┤
│                1 │
└──────────────────┘



One row in fact_content_daily_performance = one day's GSC/GA4 performance
snapshot for one piece of content, for one client (grain: report_date ×
client_hash_id × content_hash_id). Since my lane's decision is about a page,
not a single day, I aggregate these daily rows up to one row per
(client_hash_id, content_hash_id) within a chosen month.

Time window: month=2026-03 — a mid-panel month, not the _sample (which is
June 2026, the sealed final month). Using the final month to build label
logic would mean training on the exact outcome window I'm trying to predict.

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# See the actual columns in the main fact table
print(con.sql(f"DESCRIBE SELECT * FROM read_parquet('{rel}/fact_content_daily_performance/**/*.parquet') LIMIT 1"))



┌────────────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│    column_name     │ column_type │  null   │   key   │ default │  extra  │
│      varchar       │   varchar   │ varchar │ varchar │ varchar │ varchar │
├────────────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ report_date        │ DATE        │ YES     │ NULL    │ NULL    │ NULL    │
│ client_hash_id     │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ content_hash_id    │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ client_has_gsc     │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ client_has_ga4     │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ gsc_data_available │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ ga4_data_available │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ gsc_impressions    │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ gsc_clicks         │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │

Features (predictive inputs, from the first half of the month — before my
decision point): gsc_clicks, gsc_impressions, gsc_sum_position,
scroll_events, sessions_ai.

Label / proxy: whether a page's clicks in the second half of the month
fell below its first-half clicks — is_declining_label. Not a real
"business outcome," just a within-month proxy for direction.

Context (used for joining/grouping, not as model input): client_hash_id,
content_hash_id, report_date, month.

Excluded (with why): rows where gsc_data_available IS NOT TRUE. Some
clients haven't connected Search Console for the full window (dim_clients
shows several with gsc_data_start = None) — treating their missing/zero
metrics as "zero performance" would be wrong; they're missing data, not
bad data.

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [10]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
from datasets import load_dataset

dim_clients = load_dataset("FlyRank/internship-warehouse", "dim_clients", split="train", token=token).to_pandas()
print(dim_clients.shape)
print(dim_clients.head())

# Query 2: slice row count + date span for month=2026-03
print(con.sql(f"""
    SELECT COUNT(*) AS row_count, MIN(report_date) AS min_date, MAX(report_date) AS max_date
    FROM read_parquet('{rel}/fact_content_daily_performance/**/*.parquet')
    WHERE month = '2026-03'
"""))

# Query 3: availability filter with IS TRUE
print(con.sql(f"""
    SELECT
      COUNT(*) AS total_rows,
      COUNT(*) FILTER (WHERE gsc_data_available IS TRUE) AS gsc_available_rows
    FROM read_parquet('{rel}/fact_content_daily_performance/**/*.parquet')
    WHERE month = '2026-03'
"""))

# Five-feature frame + the deliberate label-derived trap
feature_frame = con.sql(f"""
WITH month_data AS (
    SELECT * FROM read_parquet('{rel}/fact_content_daily_performance/**/*.parquet')
    WHERE month = '2026-03' AND gsc_data_available IS TRUE
),
first_half AS (
    SELECT client_hash_id, content_hash_id,
           SUM(gsc_clicks) AS clicks_first_half,
           SUM(gsc_impressions) AS impressions_first_half,
           AVG(gsc_sum_position) AS avg_position_first_half,
           SUM(scroll_events) AS scroll_events_first_half,
           SUM(sessions_ai) AS ai_sessions_first_half
    FROM month_data WHERE report_date <= DATE '2026-03-15'
    GROUP BY 1,2
),
second_half AS (
    SELECT client_hash_id, content_hash_id,
           SUM(gsc_clicks) AS clicks_second_half
    FROM month_data WHERE report_date > DATE '2026-03-15'
    GROUP BY 1,2
)
SELECT f.*, s.clicks_second_half,
       CASE WHEN s.clicks_second_half < f.clicks_first_half THEN 1 ELSE 0 END AS is_declining_label
FROM first_half f JOIN second_half s USING (client_hash_id, content_hash_id)
""").df()

print(feature_frame.shape)
print(feature_frame.head())

# Honest quick score — using only first-half (pre-decision) signals
frame = feature_frame.copy()
frame["quick_pred_honest"] = (
    (frame["avg_position_first_half"] > 20) &
    (frame["clicks_first_half"] < frame["impressions_first_half"] * 0.02)
).astype(int)
honest_accuracy = (frame["quick_pred_honest"] == frame["is_declining_label"]).mean()
print("Honest accuracy:", honest_accuracy)

# THE TRAP: add a label-derived column on purpose
frame["quick_pred_leaky"] = (frame["clicks_second_half"] < frame["clicks_first_half"]).astype(int)
leaky_accuracy = (frame["quick_pred_leaky"] == frame["is_declining_label"]).mean()
print("Leaky accuracy (jumps toward perfect):", leaky_accuracy)

# Delete the leak, keep the honest number
frame = frame.drop(columns=["quick_pred_leaky"])
print("Kept, honest accuracy:", honest_accuracy)

(104, 9)
            client_hash_id is_active has_gsc_access has_ga4_access  \
0  client_04660893ae39614a      True           True           True   
1  client_05475c07ed21a83a      True          False          False   
2  client_06d356715a8ff3b6      True           True           True   
3  client_0797ff3a1fc9a6a5      True          False          False   
4  client_08a6a72ff48e62c0      True           True          False   

                  access_profile client_created_date client_updated_date  \
0                    gsc_and_ga4          2026-04-15          2026-06-27   
1  no_search_or_analytics_access          2026-04-01          2026-06-27   
2                    gsc_and_ga4          2026-03-23          2026-07-05   
3  no_search_or_analytics_access          2025-05-26          2026-06-27   
4                       gsc_only          2025-05-26          2026-06-27   

  gsc_data_start ga4_data_start  
0           None     2026-05-22  
1           None           None  
2     2026-

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌────────────┬────────────────────┐
│ total_rows │ gsc_available_rows │
│   int64    │       int64        │
├────────────┼────────────────────┤
│    9841378 │            3611061 │
└────────────┴────────────────────┘



FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

(141467, 9)
            client_hash_id           content_hash_id  clicks_first_half  \
0  client_73cda7b4e4f265ea  content_7a105f548d9c6916                6.0   
1  client_73cda7b4e4f265ea  content_a3ea9792f793ec72                0.0   
2  client_73cda7b4e4f265ea  content_36c36abc7650d7af                3.0   
3  client_73cda7b4e4f265ea  content_a7da352b73b02668                8.0   
4  client_73cda7b4e4f265ea  content_1855a661b4d36130                1.0   

   impressions_first_half  avg_position_first_half  scroll_events_first_half  \
0                  4173.0              1742.933333                       NaN   
1                   245.0                66.733333                       NaN   
2                  3705.0              1555.533333                       NaN   
3                  2440.0              1199.000000                       NaN   
4                   240.0                60.333333                       NaN   

   ai_sessions_first_half  clicks_second_half  is_declin

The five features (clicks/impressions/position/scroll/AI-sessions from the
first half) are each knowable at the decision moment because they're all
logged before the review point in the month. The leaky version used
clicks_second_half directly — which is literally what the label is defined
from — so its "accuracy" jumping near 1.0 isn't a real model, it's just
restating the label. I deleted that column and kept the honest, lower
number.

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

In [11]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


This is an unbalanced panel — dim_clients shows several clients with
gsc_data_start = None, meaning they have no GSC history at all for part or
all of this window, so month=2026-03 completeness differs by client. This
data also only captures search/AI-referral traffic behavior — it says
nothing about revenue or conversions, so it can support a review-priority
decision, not a business-impact claim.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.